# DSC2026 — Prism Private Inference v5 — Source-Faithful + Last-Token Logits

This notebook was rebuilt after inspecting the **actual fine-tuning source** in
`results/from_drive/fine_tune_source_code/`.

It supersedes the earlier reverse-engineered v1–v4 runners.

## Exact recovered contract

- base: `infgrad/Prism-Qwen3.5-Reranker-2B`
- frozen base: **fp16**
- LoRA: `r=16`, `alpha=32`, `dropout=0.05`, all linear layers
- trainable LoRA tensors: **fp32**
- exact short Prism system prompt
- exact Vietnamese-legal retrieval instruction used during fine-tuning
- left padding
- **body-only truncation** after reserving prefix + assistant suffix
- max length: **1024**
- score: `logit("yes") - logit("no")`
- document pooling: **max over two passages**

## Safe performance optimization

The fine-tune source computes full-vocabulary logits for every sequence position
and only then slices the final token. Transformers 5.17 supports
`logits_to_keep=1`, so v5 computes the same final-token logits without projecting
every earlier hidden state through the huge vocabulary head. It also disables
`use_cache`, which is unnecessary for one-shot relevance scoring.

Before private scoring, each GPU compares the original source path against the
optimized path on the same examples and aborts if score parity fails.


## 1. Install dependencies

In [ ]:
import sys, subprocess

def pip(*args, check=True):
    cmd = [sys.executable, '-m', 'pip', *args]
    print('$', ' '.join(cmd), flush=True)
    return subprocess.run(cmd, check=check)

pip('uninstall', '-y', 'torchao', check=False)
pip(
    'install', '-q', '--no-cache-dir', '-U',
    'transformers==5.17.0',
    'peft==0.21.0',
    'accelerate',
    'safetensors',
    'sentencepiece',
    'einops',
)
pip('install', '-q', '--no-build-isolation', 'causal-conv1d')
pip('install', '-q', 'flash-linear-attention==0.5.2')

verify = r"""
import torch, transformers, peft
from transformers import Qwen3_5TextConfig, Qwen3_5ForCausalLM
from fla.ops.gated_delta_rule import chunk_gated_delta_rule
from causal_conv1d import causal_conv1d_fn
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('peft:', peft.__version__)
print('Qwen3.5: PASS')
print('FLA:', chunk_gated_delta_rule.__module__, chunk_gated_delta_rule.__name__)
print('causal_conv1d:', causal_conv1d_fn.__module__, causal_conv1d_fn.__name__)
"""
subprocess.check_call([sys.executable, '-c', verify])


## 2. Verify 2×T4 and locate inputs

In [ ]:
import os, sys
from pathlib import Path
import torch

print('Python:', sys.executable)
print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available(), '| GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name} | {p.total_memory/2**30:.1f} GiB')

assert torch.cuda.device_count() >= 2, 'Select Kaggle accelerator: GPU T4 x2'

INPUT = Path('/kaggle/input')
ck = list(INPUT.rglob('best_state.pt'))
wl = list(INPUT.rglob('PRISM_PRIVATE_PASSAGES.pkl'))
print('checkpoint:', ck)
print('workload:', wl)
assert len(ck) == 1, f'Expected one best_state.pt, got {len(ck)}'
assert len(wl) == 1, f'Expected one PRISM_PRIVATE_PASSAGES.pkl, got {len(wl)}'

CHECKPOINT = ck[0]
WORKLOAD = wl[0]
OUT = Path('/kaggle/working/prism_private_v5')
OUT.mkdir(parents=True, exist_ok=True)
print('CHECKPOINT:', CHECKPOINT)
print('WORKLOAD:', WORKLOAD)
print('OUT:', OUT)


## 3. Build the **source-faithful** token-length manifest

The original fine-tune code reserves chat prefix + assistant suffix first and
truncates **only the body**. This cell regenerates lengths from that exact rule.
Do not reuse the v3/v4 manifest.


In [ ]:
import pickle, hashlib, time
from transformers import AutoTokenizer
from transformers.utils import logging as hf_logging

hf_logging.disable_progress_bar()
BASE_MODEL = 'infgrad/Prism-Qwen3.5-Reranker-2B'
MAX_LENGTH = 1024
SYSTEM_PROMPT = (
    'Judge whether the Document meets the requirements based on the '
    'Query and the Instruct provided.'
)
INSTRUCTION = (
    'Given a Vietnamese legal question, determine whether the Document '
    'contains the answer to the Query'
)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(8 << 20), b''):
            h.update(b)
    return h.hexdigest()

tok = AutoTokenizer.from_pretrained(BASE_MODEL, padding_side='left')
prefix = (
    f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
    '<|im_start|>user\n'
)
suffix = (
    '<|im_end|>\n'
    '<|im_start|>assistant\n'
    '<think>\n\n</think>\n\n'
)
prefix_ids = tok.encode(prefix, add_special_tokens=False)
suffix_ids = tok.encode(suffix, add_special_tokens=False)
budget = max(1, MAX_LENGTH - len(prefix_ids) - len(suffix_ids))
print(
    'exact prompt token budget:',
    f'prefix={len(prefix_ids)} body<={budget} suffix={len(suffix_ids)} total<={MAX_LENGTH}'
)

obj = pickle.loads(WORKLOAD.read_bytes())
queries = obj['queries']
compact = []
for qid in sorted(queries):
    row = queries[qid]
    for doc_id in row['docs']:
        for passage_idx, passage in enumerate(row['docs'][doc_id]):
            compact.append((str(qid), str(doc_id), int(passage_idx), row['question'], passage))

manifest_path = OUT / 'PRISM_SOURCE_EXACT_LENGTH_MANIFEST.pkl'
records = []
chunk_size = 512
for start in range(0, len(compact), chunk_size):
    chunk = compact[start:start+chunk_size]
    contents = [
        f'<Instruct>: {INSTRUCTION}\n<Query>: {x[3]}\n<Document>: {x[4]}'
        for x in chunk
    ]
    enc = tok(
        contents,
        add_special_tokens=False,
        truncation=True,
        max_length=budget,
        padding=False,
        return_length=True,
        return_attention_mask=False,
    )
    body_lengths = list(map(int, enc['length']))
    for j, ((qid, doc_id, passage_idx, _, _), blen) in enumerate(zip(chunk, body_lengths)):
        rid = start + j
        total_len = len(prefix_ids) + blen + len(suffix_ids)
        records.append((rid, qid, doc_id, passage_idx, total_len))
    done = min(start + len(chunk), len(compact))
    if start == 0 or (start // chunk_size) % 25 == 0:
        print(f'length scan {done:,}/{len(compact):,} ({100*done/len(compact):.1f}%)', flush=True)

records.sort(key=lambda r: r[4], reverse=True)
manifest = {
    'schema': 'manual.prism_source_exact_length_manifest.v5',
    'base_model': BASE_MODEL,
    'max_length': MAX_LENGTH,
    'workload_sha256': sha256_file(WORKLOAD),
    'records': records,
}
manifest_path.write_bytes(pickle.dumps(manifest, protocol=5))

lengths = sorted(r[4] for r in records)
def pct(p):
    return lengths[round((len(lengths)-1)*p)]

print('='*88)
print('MANIFEST:', manifest_path)
print('passages:', f'{len(lengths):,}')
print('length min/p25/p50/p75/p90/p95/max:', min(lengths), pct(.25), pct(.50), pct(.75), pct(.90), pct(.95), max(lengths))
print('truncated@1024:', sum(x >= 1024 for x in lengths), f'({100*sum(x >= 1024 for x in lengths)/len(lengths):.2f}%)')
print('='*88)


## 4. Write the source-faithful worker

In [ ]:
from pathlib import Path

WORKER = '#!/usr/bin/env python\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport pickle\nimport time\nfrom pathlib import Path\n\nos.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")\nos.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")\nos.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")\nos.environ.setdefault("TOKENIZERS_PARALLELISM", "false")\n\nimport numpy as np\nimport torch\n\nBASE_MODEL = "infgrad/Prism-Qwen3.5-Reranker-2B"\nSYSTEM_PROMPT = (\n    "Judge whether the Document meets the requirements based on the "\n    "Query and the Instruct provided."\n)\nINSTRUCTION = (\n    "Given a Vietnamese legal question, determine whether the Document "\n    "contains the answer to the Query"\n)\n\n\ndef atomic_pickle(path: Path, obj) -> None:\n    path = Path(path)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    tmp.write_bytes(pickle.dumps(obj, protocol=5))\n    os.replace(tmp, path)\n\n\ndef atomic_json(path: Path, obj) -> None:\n    path = Path(path)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    tmp.write_text(json.dumps(obj, indent=2), encoding="utf-8")\n    os.replace(tmp, path)\n\n\ndef infer_lora_contract(state):\n    a_keys = [k for k in state if ".lora_A." in k]\n    if not a_keys:\n        raise RuntimeError("No LoRA A tensors found in checkpoint")\n    ranks = {int(state[k].shape[0]) for k in a_keys}\n    if len(ranks) != 1:\n        raise RuntimeError(f"Mixed LoRA ranks: {ranks}")\n    rank = next(iter(ranks))\n    targets = sorted({k.split(".lora_A.")[0].split(".")[-1] for k in a_keys})\n    return rank, targets, len(a_keys)\n\n\ndef find_wrapped_implementation(fn):\n    seen = set()\n    cur = fn\n    for _ in range(12):\n        if cur is None or id(cur) in seen:\n            break\n        seen.add(id(cur))\n        closure = getattr(cur, "__closure__", None)\n        code = getattr(cur, "__code__", None)\n        if closure and code:\n            for name, cell in zip(code.co_freevars, closure):\n                if name == "implementation":\n                    try:\n                        impl = cell.cell_contents\n                        return f"{impl.__module__}.{getattr(impl, \'__name__\', type(impl).__name__)}"\n                    except Exception:\n                        pass\n        cur = getattr(cur, "__wrapped__", None)\n    return None\n\n\ndef report_kernel_resolution():\n    try:\n        import transformers.models.qwen3_5.modeling_qwen3_5 as mq\n        gated = find_wrapped_implementation(mq.torch_chunk_gated_delta_rule)\n        conv = find_wrapped_implementation(mq.causal_conv1d_fn)\n        print("kernel resolver:", flush=True)\n        print(f"  gated_delta_rule -> {gated or \'implementation not introspectable\'}", flush=True)\n        print(f"  causal_conv1d    -> {conv or \'implementation not introspectable\'}", flush=True)\n    except Exception as e:\n        print(f"kernel resolver diagnostic unavailable: {e}", flush=True)\n\n\ndef load_model(checkpoint: Path, gpu: int):\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    from transformers.utils import logging as hf_logging\n    from peft import LoraConfig, TaskType, get_peft_model\n\n    hf_logging.disable_progress_bar()\n    torch.cuda.set_device(gpu)\n    device = f"cuda:{gpu}"\n    report_kernel_resolution()\n\n    print(f"loading checkpoint: {checkpoint}", flush=True)\n    obj = torch.load(checkpoint, map_location="cpu", weights_only=True)\n    state = obj.get("state_dict", obj)\n    rank, targets, n_a = infer_lora_contract(state)\n    print(\n        f"LoRA checkpoint | r={rank} alpha=32 dropout=0.05 "\n        f"A_modules={n_a} tensors={len(state)}",\n        flush=True,\n    )\n    print("targets=" + ",".join(targets), flush=True)\n\n    tok = AutoTokenizer.from_pretrained(BASE_MODEL, padding_side="left")\n    tok.padding_side = "left"\n    if tok.pad_token_id is None:\n        tok.pad_token = tok.eos_token\n\n    print(f"loading base fp16: {BASE_MODEL}", flush=True)\n    base = AutoModelForCausalLM.from_pretrained(\n        BASE_MODEL, dtype=torch.float16, low_cpu_mem_usage=True\n    )\n\n    cfg = LoraConfig(\n        r=rank,\n        lora_alpha=32,\n        lora_dropout=0.05,\n        target_modules=targets,\n        bias="none",\n        task_type=TaskType.CAUSAL_LM,\n    )\n    model = get_peft_model(base, cfg)\n\n    trainable = 0\n    for _, param in model.named_parameters():\n        if param.requires_grad:\n            param.data = param.data.float()\n            trainable += param.numel()\n\n    incompat = model.load_state_dict(state, strict=False)\n    if incompat.unexpected_keys:\n        raise RuntimeError(f"Unexpected checkpoint keys: {incompat.unexpected_keys[:20]}")\n\n    model.eval().to(device)\n    yes_id = tok.encode("yes", add_special_tokens=False)[-1]\n    no_id = tok.encode("no", add_special_tokens=False)[-1]\n\n    active_a = sum(1 for name, _ in model.named_parameters() if ".lora_A." in name)\n    if active_a != n_a:\n        raise RuntimeError(f"LoRA module mismatch: checkpoint={n_a}, model={active_a}")\n\n    alloc = torch.cuda.memory_allocated(gpu) / 2**30\n    reserved = torch.cuda.memory_reserved(gpu) / 2**30\n    print(\n        f"model ready | {device} | trainable={trainable/1e6:.1f}M | "\n        f"VRAM={alloc:.2f}/{reserved:.2f} GiB allocated/reserved",\n        flush=True,\n    )\n    return model, tok, yes_id, no_id, device\n\n\ndef prompt_parts(tokenizer):\n    prefix = (\n        f"<|im_start|>system\\n{SYSTEM_PROMPT}<|im_end|>\\n"\n        "<|im_start|>user\\n"\n    )\n    suffix = (\n        "<|im_end|>\\n"\n        "<|im_start|>assistant\\n"\n        "<think>\\n\\n</think>\\n\\n"\n    )\n    return (\n        tokenizer.encode(prefix, add_special_tokens=False),\n        tokenizer.encode(suffix, add_special_tokens=False),\n    )\n\n\ndef make_input_ids(tokenizer, questions, passages, max_length):\n    prefix_ids, suffix_ids = prompt_parts(tokenizer)\n    budget = max(1, max_length - len(prefix_ids) - len(suffix_ids))\n    contents = [\n        f"<Instruct>: {INSTRUCTION}\\n<Query>: {q}\\n<Document>: {p}"\n        for q, p in zip(questions, passages)\n    ]\n    body = tokenizer(\n        contents,\n        add_special_tokens=False,\n        truncation=True,\n        max_length=budget,\n    )\n    return [prefix_ids + ids + suffix_ids for ids in body["input_ids"]]\n\n\ndef make_batch(tokenizer, questions, passages, max_length, device):\n    tokenizer.padding_side = "left"\n    input_ids = make_input_ids(tokenizer, questions, passages, max_length)\n    batch = tokenizer.pad({"input_ids": input_ids}, padding=True, return_tensors="pt")\n    return {k: v.to(device, non_blocking=True) for k, v in batch.items()}\n\n\n@torch.inference_mode()\ndef score_source_exact(model, tokenizer, yes_id, no_id, device, questions, passages, max_length):\n    batch = make_batch(tokenizer, questions, passages, max_length, device)\n    logits = model(**batch, return_dict=True).logits[:, -1, :]\n    return (logits[:, yes_id] - logits[:, no_id]).float()\n\n\n@torch.inference_mode()\ndef score_optimized(model, tokenizer, yes_id, no_id, device, questions, passages, max_length):\n    batch = make_batch(tokenizer, questions, passages, max_length, device)\n    logits = model(\n        **batch,\n        return_dict=True,\n        logits_to_keep=1,\n        use_cache=False,\n    ).logits[:, -1, :]\n    return (logits[:, yes_id] - logits[:, no_id]).float()\n\n\ndef records_to_inputs(records, queries):\n    questions, passages = [], []\n    for _, qid, doc_id, passage_idx, _ in records:\n        row = queries[qid]\n        questions.append(row["question"])\n        passages.append(row["docs"][doc_id][passage_idx])\n    return questions, passages\n\n\ndef correctness_preflight(model, tokenizer, yes_id, no_id, device, records, queries, max_length):\n    sample = records[:4]\n    q, p = records_to_inputs(sample, queries)\n\n    torch.cuda.synchronize()\n    t0 = time.perf_counter()\n    ref = score_source_exact(model, tokenizer, yes_id, no_id, device, q, p, max_length)\n    torch.cuda.synchronize()\n    t_ref = time.perf_counter() - t0\n\n    torch.cuda.synchronize()\n    t0 = time.perf_counter()\n    opt = score_optimized(model, tokenizer, yes_id, no_id, device, q, p, max_length)\n    torch.cuda.synchronize()\n    t_opt = time.perf_counter() - t0\n\n    ref_cpu = ref.detach().float().cpu()\n    opt_cpu = opt.detach().float().cpu()\n    diff = (ref_cpu - opt_cpu).abs()\n    max_abs = float(diff.max())\n    mean_abs = float(diff.mean())\n    same_order = bool(\n        torch.equal(\n            torch.argsort(ref_cpu, descending=True),\n            torch.argsort(opt_cpu, descending=True),\n        )\n    )\n    print(\n        "SOURCE-PARITY PREFLIGHT | "\n        f"max_abs={max_abs:.6f} mean_abs={mean_abs:.6f} "\n        f"same_order={same_order} | full_logits={t_ref:.2f}s "\n        f"optimized={t_opt:.2f}s speedup={t_ref/max(t_opt,1e-9):.2f}x",\n        flush=True,\n    )\n    if max_abs > 0.02 or not same_order:\n        raise RuntimeError(\n            "Optimized last-token path does not reproduce source-faithful scores closely enough"\n        )\n\n\ndef autotune(model, tokenizer, yes_id, no_id, device, records, queries, max_length, candidates, sample_n):\n    n = min(sample_n, len(records))\n    center = max(0, min(len(records) - n, len(records) // 4 - n // 2))\n    sample = records[center:center+n]\n    lengths = [int(r[4]) for r in sample]\n    print(\n        f"AUTOTUNE sample={len(sample)} len={min(lengths)}..{max(lengths)} "\n        f"median={int(np.median(lengths))}",\n        flush=True,\n    )\n\n    warm = sample[:min(4, len(sample))]\n    q, p = records_to_inputs(warm, queries)\n    _ = score_optimized(model, tokenizer, yes_id, no_id, device, q, p, max_length)\n    torch.cuda.synchronize()\n\n    rows = []\n    for bs in candidates:\n        torch.cuda.empty_cache()\n        torch.cuda.reset_peak_memory_stats()\n        ok = True\n        t0 = time.perf_counter()\n        try:\n            for i in range(0, len(sample), bs):\n                chunk = sample[i:i+bs]\n                q, p = records_to_inputs(chunk, queries)\n                _ = score_optimized(model, tokenizer, yes_id, no_id, device, q, p, max_length)\n            torch.cuda.synchronize()\n        except torch.cuda.OutOfMemoryError:\n            ok = False\n            torch.cuda.empty_cache()\n        dt = time.perf_counter() - t0\n        peak = torch.cuda.max_memory_allocated() / 2**30\n        if ok:\n            pps = len(sample) / dt\n            rows.append({\n                "batch_size": int(bs),\n                "seconds": dt,\n                "prompts_per_second": pps,\n                "peak_vram_gib": peak,\n            })\n            print(\n                f"AUTOTUNE bs={bs:>3} | {dt:7.2f}s | {pps:7.2f} passages/s | peak={peak:.2f} GiB",\n                flush=True,\n            )\n        else:\n            print(f"AUTOTUNE bs={bs:>3} | OOM | peak={peak:.2f} GiB", flush=True)\n\n    if not rows:\n        raise RuntimeError("All autotune batch sizes OOM")\n    best = max(rows, key=lambda x: x["prompts_per_second"])\n    rep_len = max(lengths)\n    token_budget = int(best["batch_size"]) * rep_len\n    print(\n        f"AUTOTUNE BEST bs={best[\'batch_size\']} pps={best[\'prompts_per_second\']:.2f} "\n        f"rep_len={rep_len} token_budget={token_budget:,}",\n        flush=True,\n    )\n    return token_budget, rows\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--gpu", type=int, required=True)\n    ap.add_argument("--shard", type=int, required=True)\n    ap.add_argument("--num-shards", type=int, default=2)\n    ap.add_argument("--workload", type=Path, required=True)\n    ap.add_argument("--manifest", type=Path, required=True)\n    ap.add_argument("--checkpoint", type=Path, required=True)\n    ap.add_argument("--output", type=Path, required=True)\n    ap.add_argument("--report", type=Path, required=True)\n    ap.add_argument("--max-length", type=int, default=1024)\n    ap.add_argument("--max-batch", type=int, default=128)\n    ap.add_argument("--autotune-sample", type=int, default=64)\n    ap.add_argument("--autotune-batches", default="8,16,32,64")\n    ap.add_argument("--save-every-batches", type=int, default=20)\n    ap.add_argument("--log-every-batches", type=int, default=5)\n    args = ap.parse_args()\n\n    print(f"worker start | gpu={args.gpu} shard={args.shard}/{args.num_shards}", flush=True)\n    model, tok, yes_id, no_id, device = load_model(args.checkpoint, args.gpu)\n\n    workload_obj = pickle.loads(args.workload.read_bytes())\n    queries = workload_obj["queries"]\n    manifest = pickle.loads(args.manifest.read_bytes())\n    records_all = manifest["records"]\n    records = [rec for i, rec in enumerate(records_all) if i % args.num_shards == args.shard]\n\n    saved = pickle.loads(args.output.read_bytes()) if args.output.is_file() else {}\n    saved = {int(k): float(v) for k, v in saved.items()}\n    remaining = [rec for rec in records if int(rec[0]) not in saved]\n    remaining.sort(key=lambda r: int(r[4]), reverse=True)\n    print(\n        f"records | shard={len(records):,} cached={len(saved):,} remaining={len(remaining):,}",\n        flush=True,\n    )\n    if not remaining:\n        print("shard already complete", flush=True)\n        return\n\n    correctness_preflight(model, tok, yes_id, no_id, device, remaining, queries, args.max_length)\n    candidates = [int(x) for x in args.autotune_batches.split(",") if x.strip()]\n    token_budget, bench = autotune(\n        model, tok, yes_id, no_id, device, remaining, queries, args.max_length,\n        candidates, args.autotune_sample,\n    )\n\n    total = len(remaining)\n    i = done = batch_no = 0\n    start = time.perf_counter()\n    raw_tokens_done = padded_tokens_done = 0\n    max_batch = int(args.max_batch)\n\n    while i < total:\n        longest = max(1, int(remaining[i][4]))\n        bs = min(max_batch, max(1, token_budget // longest), total - i)\n        batch_records = remaining[i:i+bs]\n        q, p = records_to_inputs(batch_records, queries)\n\n        try:\n            score = score_optimized(model, tok, yes_id, no_id, device, q, p, args.max_length)\n        except torch.cuda.OutOfMemoryError:\n            torch.cuda.empty_cache()\n            if bs <= 1:\n                raise\n            max_batch = min(max_batch, max(1, bs // 2))\n            token_budget = min(token_budget, max_batch * longest)\n            print(\n                f"CUDA OOM | len={longest} bs={bs} -> max_batch={max_batch} "\n                f"token_budget={token_budget:,}; retry",\n                flush=True,\n            )\n            continue\n\n        vals = score.detach().float().cpu().tolist()\n        for rec, value in zip(batch_records, vals):\n            if not np.isfinite(value):\n                raise RuntimeError(f"non-finite score rid={rec[0]}: {value}")\n            saved[int(rec[0])] = float(value)\n\n        raw_tokens_done += sum(int(r[4]) for r in batch_records)\n        padded_tokens_done += longest * len(batch_records)\n        i += len(batch_records)\n        done += len(batch_records)\n        batch_no += 1\n\n        if batch_no % args.save_every_batches == 0:\n            atomic_pickle(args.output, saved)\n\n        if batch_no == 1 or batch_no % args.log_every_batches == 0 or done == total:\n            elapsed = time.perf_counter() - start\n            pps = done / max(elapsed, 1e-9)\n            eta = (total - done) / max(pps, 1e-9) / 60\n            eff = raw_tokens_done / max(padded_tokens_done, 1)\n            peak = torch.cuda.max_memory_allocated() / 2**30\n            print(\n                f"progress {done:,}/{total:,} ({100*done/total:5.1f}%) | "\n                f"{pps:7.2f} passages/s | ETA {eta:6.1f}m | "\n                f"len={longest:4d} bs={len(batch_records):3d} | "\n                f"pad_eff={100*eff:5.1f}% | peak={peak:.2f} GiB",\n                flush=True,\n            )\n\n    atomic_pickle(args.output, saved)\n    elapsed = time.perf_counter() - start\n    report = {\n        "schema": "manual.prism_sourcefaithful_worker.v5",\n        "gpu": args.gpu,\n        "shard": args.shard,\n        "records": len(records),\n        "scoring_seconds": elapsed,\n        "passages_per_second": total / max(elapsed, 1e-9),\n        "padding_efficiency": raw_tokens_done / max(padded_tokens_done, 1),\n        "token_budget": token_budget,\n        "max_batch_final": max_batch,\n        "autotune": bench,\n    }\n    atomic_json(args.report, report)\n    print(\n        f"COMPLETE | {len(saved):,}/{len(records):,} | {elapsed/60:.1f}m | "\n        f"{report[\'passages_per_second\']:.2f} passages/s",\n        flush=True,\n    )\n\n\nif __name__ == "__main__":\n    main()\n'

WORKER_PATH = Path('/kaggle/working/prism_worker_source_exact_v5.py')
WORKER_PATH.write_text(WORKER, encoding='utf-8')
print('worker:', WORKER_PATH)
print('bytes:', WORKER_PATH.stat().st_size)


## 5. Source-parity preflight + both T4 workers

First inspect these lines:

```text
SOURCE-PARITY PREFLIGHT | ... same_order=True | full_logits=... optimized=... speedup=...x
AUTOTUNE ...
AUTOTUNE BEST ...
```

Only after parity passes does private scoring continue.


In [ ]:
import os, sys, time, queue, threading, subprocess

for pattern in (
    '/kaggle/working/prism_worker.py',
    '/kaggle/working/prism_bucket_worker_v3.py',
    '/kaggle/working/prism_bucket_worker_v4.py',
    '/kaggle/working/prism_worker_source_exact_v5.py',
):
    subprocess.run(['pkill', '-f', pattern], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)

env = os.environ.copy()
env['TOKENIZERS_PARALLELISM'] = 'false'
env['PYTHONUNBUFFERED'] = '1'
env['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
env['TRANSFORMERS_VERBOSITY'] = 'error'
env['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'
env.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

events = queue.Queue()
procs = []

def pump(gpu, proc):
    try:
        for line in proc.stdout:
            events.put((gpu, line.rstrip('\n')))
    finally:
        events.put((gpu, None))

for gpu in (0, 1):
    cmd = [
        sys.executable, str(WORKER_PATH),
        '--gpu', str(gpu),
        '--shard', str(gpu),
        '--num-shards', '2',
        '--workload', str(WORKLOAD),
        '--manifest', str(manifest_path),
        '--checkpoint', str(CHECKPOINT),
        '--output', str(OUT / f'passage_scores_gpu{gpu}.pkl'),
        '--report', str(OUT / f'worker_report_gpu{gpu}.json'),
        '--max-length', '1024',
        '--max-batch', '128',
        '--autotune-sample', '64',
        '--autotune-batches', '8,16,32,64',
        '--save-every-batches', '20',
        '--log-every-batches', '5',
    ]
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    procs.append(p)
    threading.Thread(target=pump, args=(gpu, p), daemon=True).start()
    print(f'[MAIN] launched GPU{gpu} pid={p.pid}', flush=True)

finished = 0
try:
    while finished < 2:
        try:
            gpu, line = events.get(timeout=1)
        except queue.Empty:
            continue
        if line is None:
            finished += 1
        else:
            print(f'[GPU{gpu}] {line}', flush=True)
except KeyboardInterrupt:
    print('\n[MAIN] interrupted; terminating workers...', flush=True)
    for p in procs:
        if p.poll() is None:
            p.terminate()
    raise

codes = [p.wait() for p in procs]
print('[MAIN] return codes:', codes)
if codes != [0, 0]:
    raise RuntimeError(f'Worker failure: {codes}')
print('[MAIN] BOTH SHARDS COMPLETE')


## 6. Merge passages → exact document `top2_max` cache

In [ ]:
import pickle, json, hashlib
from pathlib import Path

workload_obj = pickle.loads(WORKLOAD.read_bytes())
queries = workload_obj['queries']
manifest = pickle.loads(manifest_path.read_bytes())
records = manifest['records']

passage_scores = {}
for gpu in (0, 1):
    p = OUT / f'passage_scores_gpu{gpu}.pkl'
    assert p.is_file(), p
    part = {int(k): float(v) for k, v in pickle.loads(p.read_bytes()).items()}
    overlap = set(passage_scores) & set(part)
    assert not overlap, list(overlap)[:10]
    passage_scores.update(part)

expected = {int(r[0]) for r in records}
assert set(passage_scores) == expected, (
    f'passage coverage mismatch: missing={len(expected-set(passage_scores))} '
    f'extra={len(set(passage_scores)-expected)}'
)

merged = {}
for rid, qid, doc_id, passage_idx, token_len in records:
    score = passage_scores[int(rid)]
    qrow = merged.setdefault(str(qid), {})
    d = str(doc_id)
    if d not in qrow or score > qrow[d]:
        qrow[d] = score

missing, extra = [], []
pairs = 0
for qid, row in queries.items():
    exp = set(map(str, row['docs']))
    got = set(merged.get(str(qid), {}))
    pairs += len(exp)
    missing.extend((str(qid), d) for d in exp-got)
    extra.extend((str(qid), d) for d in got-exp)

assert len(merged) == 2080, len(merged)
assert not missing, missing[:20]
assert not extra, extra[:20]

final = Path('/kaggle/working/prism_private_scores.pkl')
final.write_bytes(pickle.dumps(merged, protocol=5))
sha = hashlib.sha256(final.read_bytes()).hexdigest()

worker_reports = {}
for gpu in (0, 1):
    rp = OUT / f'worker_report_gpu{gpu}.json'
    if rp.exists():
        worker_reports[str(gpu)] = json.loads(rp.read_text())

vals = [v for row in merged.values() for v in row.values()]
report = {
    'schema': 'manual.kaggle_prism_source_exact_2xt4.v5',
    'queries': len(merged),
    'document_pairs': pairs,
    'passages': len(passage_scores),
    'score_min': min(vals),
    'score_max': max(vals),
    'sha256': sha,
    'base': 'infgrad/Prism-Qwen3.5-Reranker-2B',
    'lora_r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,
    'mode': 'top2_max',
    'max_length': 1024,
    'optimization': 'logits_to_keep=1,use_cache=False',
    'workers': worker_reports,
}
report_path = Path('/kaggle/working/PRISM_SOURCE_EXACT_REPORT.json')
report_path.write_text(json.dumps(report, indent=2), encoding='utf-8')

print('='*88)
print('VALIDATION PASS')
print('queries:', len(merged))
print('document pairs:', f'{pairs:,}')
print('passages:', f'{len(passage_scores):,}')
print('sha256:', sha)
print('FINAL:', final)
print('REPORT:', report_path)
print('='*88)


## 7. After downloading

Download `/kaggle/working/prism_private_scores.pkl` and put it at:

`results/manual/huy_private_prism_v1/prism_private_scores.pkl`

Then run the existing `materialize_private_prism_d1_v1.py --arm prism_score_rank`.
